In [0]:
from pyspark.sql import functions as F

# Read Silver data
silver = spark.table(
    "workspace.retail_warehouse.silver_retail_transactions"
)

# ==========================================
# 1. DIMENSION: CUSTOMER
# ==========================================

dim_customer = (
    silver
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(
        F.max("country").alias("country")
    )
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.retail_warehouse.dim_customer"
)

# ==========================================
# 2. DIMENSION: PRODUCT
# ==========================================

dim_product = (
    silver
    .groupBy("stockcode")
    .agg(
        F.max("description").alias("description")
    )
)

dim_product.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.retail_warehouse.dim_product"
)

# ==========================================
# 3. DIMENSION: DATE
# ==========================================

dim_date = (
    silver
    .select(F.to_date("invoicedate").alias("full_date"))
    .filter(F.col("full_date").isNotNull())
    .distinct()
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("integer")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
)

dim_date.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.retail_warehouse.dim_date"
)

# ==========================================
# 4. FACT: SALES
# ==========================================

fact_sales = (
    silver
    .withColumn(
        "date_key",
        F.date_format("invoicedate", "yyyyMMdd").cast("integer")
    )
    .select(
        "invoice",
        "stockcode",
        "customer_id",
        "date_key",
        "invoicedate",
        "quantity",
        F.col("price").alias("unit_price"),
        "total_amount",
        "transaction_type",
        "country"
    )
)

fact_sales.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.retail_warehouse.fact_sales"
)

# ==========================================
# 5. GOLD VALIDATION
# ==========================================

tables = [
    "dim_customer",
    "dim_product",
    "dim_date",
    "fact_sales"
]

print("=" * 55)
print("GOLD DATA WAREHOUSE CREATED")
print("=" * 55)

for table in tables:
    df = spark.table(f"workspace.retail_warehouse.{table}")
    print(f"{table}: {df.count():,} records")

print("\nFACT SALES SAMPLE:")
display(
    spark.table("workspace.retail_warehouse.fact_sales").limit(10)
)

GOLD DATA WAREHOUSE CREATED
dim_customer: 5,942 records
dim_product: 5,304 records
dim_date: 604 records
fact_sales: 1,033,031 records

FACT SALES SAMPLE:


invoice,stockcode,customer_id,date_key,invoicedate,quantity,unit_price,total_amount,transaction_type,country
489436,21755,13078,20091201,2009-12-01T09:06:00.000Z,18,5.45,98.10,SALE,United Kingdom
489436,22142,13078,20091201,2009-12-01T09:06:00.000Z,12,1.45,17.40,SALE,United Kingdom
489437,20703,15362,20091201,2009-12-01T09:08:00.000Z,3,4.25,12.75,SALE,United Kingdom
489438,84031A,18102,20091201,2009-12-01T09:24:00.000Z,56,3.00,168.00,SALE,United Kingdom
489438,84519B,18102,20091201,2009-12-01T09:24:00.000Z,60,2.40,144.00,SALE,United Kingdom
489438,85183A,18102,20091201,2009-12-01T09:24:00.000Z,60,2.40,144.00,SALE,United Kingdom
489439,22064,12682,20091201,2009-12-01T09:28:00.000Z,12,1.65,19.80,SALE,France
489439,20749,12682,20091201,2009-12-01T09:28:00.000Z,2,7.95,15.90,SALE,France
489441,22321,18087,20091201,2009-12-01T09:44:00.000Z,12,0.72,8.64,SALE,United Kingdom
489442,22091,13635,20091201,2009-12-01T09:46:00.000Z,12,1.25,15.00,SALE,United Kingdom


In [0]:
%sql
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.invoice) AS total_orders,
    SUM(f.quantity) AS units_sold,
    ROUND(SUM(f.total_amount), 2) AS revenue
FROM workspace.retail_warehouse.fact_sales f
JOIN workspace.retail_warehouse.dim_date d
    ON f.date_key = d.date_key
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month;

year,month,month_name,total_orders,units_sold,revenue
2009,12,December,2330,418570,796648.50
2010,1,January,1633,374510,622479.50
2010,2,February,1969,367501,531265.36
2010,3,March,2367,488012,763247.24
2010,4,April,1891,350477,641521.05
2010,5,May,2418,362771,613270.72
2010,6,June,2216,386447,677073.87
2010,7,July,2016,301175,617365.48
2010,8,August,1877,450775,654774.39
2010,9,September,2375,476808,851105.96


In [0]:
%sql
SELECT
    p.stockcode,
    p.description,
    SUM(f.quantity) AS units_sold,
    ROUND(SUM(f.total_amount), 2) AS revenue
FROM workspace.retail_warehouse.fact_sales f
JOIN workspace.retail_warehouse.dim_product p
    ON f.stockcode = p.stockcode
GROUP BY p.stockcode, p.description
ORDER BY revenue DESC
LIMIT 10;

stockcode,description,units_sold,revenue
22423,wonky bottom/broken,24696,314045.02
DOT,DOTCOM POSTAGE,2917,309844.10
85123A,wrongly marked carton 22804,94123,248337.61
85099B,missing,94151,178400.52
47566,PARTY BUNTING,27113,147079.73
84879,damaged,78978,128550.42
22086,PAPER CHAIN KIT 50'S CHRISTMAS,34434,116298.59
POST,POSTAGE,10011,110430.41
79321,CHILLI LIGHTS,15652,79905.13
22197,SMALL POPCORN HOLDER,77969,78903.62


In [0]:
%sql
SELECT
    customer_id,
    COUNT(DISTINCT invoice) AS total_orders,
    ROUND(SUM(total_amount), 2) AS total_spending
FROM workspace.retail_warehouse.fact_sales
WHERE customer_id IS NOT NULL
GROUP BY customer_id
ORDER BY total_spending DESC
LIMIT 10;

customer_id,total_orders,total_spending
18102,153,570380.61
14646,164,523342.07
14156,202,296063.44
14911,510,265757.91
17450,61,231390.55
13694,164,190020.84
17511,85,168491.62
12415,33,143269.29
16684,65,141502.25
15061,138,124961.98
